# Template Definition

In [1]:
ROOT_FOLDER = "../dataset"

In [2]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate([
    ("system", """You are a helpful assistant expert of conceptual modeling, information extraction and UML modeling.
                  
                  You will be asked by the user to create a plant UMl model from specification text. Do so in the most
                  clear way possible, avoid class properties and assign molteplicity. 

                  Do not include attributes for classes. For example the class Book would be:

                  class Book{{}}

                  Use only bi-directional arc for relations and no description. For example a relation between
                  the class Book and the class Page, if the Book can have from one to many pages and the 
                  pages could have exactly one book, would be:

                  Book "1..1" -- "1..*" Page

                  Adapt the cardinality to each case. If the cardinality would be "0..*", the default one, omit it.

                  The plantuml has to be the class diagram. In generating the diagram perform this steps in order

                  1. Extract class from text
                  2. Extract relations form text
                  3. Assign the relation to the corresponding class
                  4. Add cardinality to the relations

                  Put everything in this order: first all classes and then all relations. In our example would be:

                  class Book{{}}
                  class Page{{}}

                  Book "1..1" -- "1..*" Page


                  Output plantuml without futher text or explaination.
                  """),
    ("user", "Transform into plant uml this specification text: {text}")
])


""" Step 1. Class
    Step 2. Association
    Step 3. Cardinaliy one by one"""

' Step 1. Class\n    Step 2. Association\n    Step 3. Cardinaliy one by one'

In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
prompt_template = PromptTemplate.from_template(
"""
You will be asked by the user to create a plant UMl model from specification text. Do so in the most
clear way possible, avoid class properties and assign molteplicity. 

Do not include attributes for classes. For example the class Book would be:

class Book{{}}

Use only bi-directional arc for relations and no description. For example a relation between
the class Book and the class Page, if the Book can have from one to many pages and the 
pages could have exactly one book, would be:

Book "1..1" -- "1..*" Page

Adapt the cardinality to each case.

The plantuml has to be the class diagram. In generating the diagram perform this steps in order

1. Extract class from text
2. Extract relations form text
3. Assign the relation to the corresponding class
4. Add cardinality to the relations

Put everything in this order: first all classes and then all relations. In our example would be:

@startuml

class Book{{}}
class Page{{}}

Book "1..1" -- "1..*" Page

@enduml

Output plantuml without futher text or explaination.

##############

The specification text is:

{text}

##############

The uml output is:
"""
)

In [4]:
from typing import Tuple
from langchain_experimental.tot.checker import ToTChecker
from langchain_experimental.tot.thought import ThoughtValidity
from lark import Lark

ModuleNotFoundError: No module named 'langchain.base_language'

In [5]:
class PlantUMLChecker(ToTChecker):
    def evaluate(
        self, problem_description: str, thoughts: Tuple[str, ...] = ()
    ) -> ThoughtValidity:
        last_thought = thoughts[-1]
        with open("../grammar.ebnf", encoding="utf-8") as grammar_file:
            parser = Lark(grammar_file.read())
        
            try:
                parser.parse(last_thought)
                return ThoughtValidity.VALID_FINAL
            except Exception as e:
                common_words = set(problem_description.lower().split()).intersection(set(last_thought.replace("{}", "").lower().split()))
                if common_words:
                    return ThoughtValidity.VALID_INTERMEDIATE
                else:
                    return ThoughtValidity.INVALID

In [6]:
from langchain_experimental.tot.base import ToTChain

In [4]:
import sys
import threading
from time import sleep
try:
    import thread
except ImportError:
    import _thread as thread

def quit_function(fn_name):
    # print to stderr, unbuffered in Python 2.
    print('{0} took too long'.format(fn_name), file=sys.stderr)
    sys.stderr.flush() # Python 3 stderr is likely buffered.
    thread.interrupt_main() # raises KeyboardInterrupt
    
def exit_after(s):
    '''
    use as decorator to exit process if 
    function takes longer than s seconds
    '''
    def outer(fn):
        def inner(*args, **kwargs):
            timer = threading.Timer(s, quit_function, args=[fn.__name__])
            timer.start()
            try:
                result = fn(*args, **kwargs)
            finally:
                timer.cancel()
            return result
        return inner
    return outer

In [5]:
import os
from tqdm import tqdm
from docx import Document

from langchain_core.tracers.context import tracing_v2_enabled

@exit_after(360)
def run_chain(chain, text):
    return chain.invoke({"text": text})

def process_subfolders_with_chain(root_folder_path, chain, type=''):
    """
    Explores subfolders of the root folder (depth 1), processes each subfolder's `text.txt`
    with the provided LangChain chain, and saves the result in a new file in the same folder.

    Args:
        root_folder_path (str): Path to the root folder.
        chain: A LangChain chain instance to process text inputs.
    """
    for subfolder_name in tqdm(os.listdir(root_folder_path)):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            text_file_path = os.path.join(subfolder_path, "description.md")

            # Recheck if `text.txt` now exists
            if os.path.isfile(text_file_path):
                with open(text_file_path, "r", encoding="utf-8") as file:
                    text = file.read()

                with tracing_v2_enabled():
                    # Call the LangChain chain with the input dictionary
                    try:
                        result = run_chain(chain, text)
                    except:
                        result = ""

                # Save the result to a new file in the same subfolder
                result_file_path = os.path.join(subfolder_path, f"result_{type}.txt")
                with open(result_file_path, "w", encoding="utf-8") as result_file:
                    result_file.write(result)

In [6]:
def delete_result_txt_files(root_folder_path):
    """
    Deletes every .txt file that starts with 'result_' in the subfolders of the root folder (depth 1).

    Args:
        root_folder_path (str): Path to the root folder.
    """
    for subfolder_name in os.listdir(root_folder_path):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            for file_name in os.listdir(subfolder_path):
                if "oss" in file_name and file_name.endswith(".txt"):
                    file_path = os.path.join(subfolder_path, file_name)
                    os.remove(file_path)

In [10]:
#delete_result_txt_files(ROOT_FOLDER)

# Zero Shot Open-AI

In [7]:
from dotenv import load_dotenv
assert load_dotenv()

In [8]:
MODEL_OPEN_AI = ["gpt-5-nano", "o3-mini", "gpt-4o-mini", "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"]

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display_markdown

In [14]:
def open_ai_zero(model):
    model_ = ChatOpenAI(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
def open_ai_make_example(model):
    model = ChatOpenAI(model=model)
    #chain = prompt_template | model | StrOutputParser()
    tot_chain = ToTChain(
        llm=model, checker=PlantUMLChecker(), k=20, c=6, verbose=True, verbose_llm=False
    )
    text = """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system\so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""
    input_data = {"problem_description": prompt_template.format(text=text)}
    res = tot_chain.invoke(input=input_data)
    return res

In [16]:
display_markdown(open_ai_make_example(MODEL_OPEN_AI[-1]), raw=True)



> Entering new ToTChain chain...
Starting the ToT solve procedure.


/Users/marcocalamo/anaconda3/envs/kul/lib/python3.10/site-packages/langchain/chains/llm.py:341: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Raw LLM output: ```json
[
"Create class diagram based on given specification text",
"Identify classes and their relationships",
"Assign relations to corresponding classes",
"Add cardinality to the relations",
"Organize classes and relations in a clear diagram",
"Ensure bi-directional arcs for relations"
]
```
Thought: Create class diagram based on given specification text


/Users/marcocalamo/anaconda3/envs/kul/lib/python3.10/site-packages/langchain/chains/llm.py:341: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Raw LLM output: 1. Extracted classes: AlphaInsurance, Broker, Customer, InsurancePolicy, Contract, Offer, Claim, Case, Estimator, Report
2. Extracted relations: Customer -assigns-> Broker, Broker -manages-> Customer, Customer -requests-> InsurancePolicy, Broker -assesses-> Customer, Customer -indicats-> InsurancePolicy, Broker -sends-> Case, Offer -extends-> Customer, Contract -signedBy-> Customer, Claim -submittedBy-> Customer, Company -opens-> Case, Case -sentFor-> Estimator, Claim -approvedBy-> Estimator, Estimator -issues-> Report
3. Assigning relations:  
     AlphaInsurance "1..*" -- "1..*" Broker
     AlphaInsurance "1..*" -- "0..*" Customer
     Broker "1..1" -- "0..*" Customer
     Customer "1..*" -- "1..*" InsurancePolicy
     Broker "1..*" -- "1..*" Case
     Customer "0..*" -- "1..1" Offer
     Contract "0..1" -- "1..1" Customer
     Claim "0..1" -- "1..1" Customer
     Company "1..1" -- "1..*" Case
     Case "1..1" -- "1..*" Estimator
     Claim "1..*" -- "1..*" Estimator


...

/Users/marcocalamo/anaconda3/envs/kul/lib/python3.10/site-packages/langchain/chains/llm.py:341: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Raw LLM output: ```json
[
"Insurance Policy{}",
"Contract{}",
"Help Desk{}",
"Account Manager{}",
"Head Office{}",
"Preliminary Contract{}"
]
```
                    Thought: Insurance Policy{}


/Users/marcocalamo/anaconda3/envs/kul/lib/python3.10/site-packages/langchain/chains/llm.py:341: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Raw LLM output: @startuml

class AlphaInsurance{}
class Customer{}
class Broker{}
class InsurancePolicy{}
class Contract{}
class HelpDesk{}
class AccountManager{}
class HeadOffice{}
class PreliminaryContract{}
class Offer{}
class ExistingCustomer{}
class Client{}
class Coverage{}
class Invoice{}
class Claim{}
class ClaimCase{}
class InsuredEvent{}
class MaterialDamage{}
class PhysicalDamage{}
class Estimator{}
class CompensationDecision{}
class Refund{}
class Database{}

AlphaInsurance "1..*" -- "1..*" Customer
Customer "1..*" -- "1..1" Broker
Customer "1..*" -- "1..*" InsurancePolicy
Customer "1..1" -- "0..1" AccountManager
Customer "1..1" -- "0..1" PreliminaryContract
PreliminaryContract "1..1" -- "0..1" Offer
Offer "1..*" -- "1..*" ExistingCustomer
PreliminaryContract "1..1" -- "1..1" Client
Client "1..1" -- "0..1" Coverage
Client "1..*" -- "1..1" Invoice
Client "1..*" -- "1..*" Claim
Claim "1..*" -- "1..*" ClaimCase
Claim "1..1" -- "1..*" InsuredEvent
ClaimCase "1..*" -- "1..*" Mat

TypeError: the JSON object must be str, bytes or bytearray, not list

In [ ]:
for model in MODEL_OPEN_AI:
    print(f"Zero shot with {model}")
    open_ai_zero(model)

Zero shot with gpt-5-nano


100%|██████████| 48/48 [47:50<00:00, 59.81s/it]


Zero shot with o3-mini


100%|██████████| 48/48 [30:50<00:00, 38.55s/it]


Zero shot with gpt-4o-mini


...

100%|██████████| 48/48 [04:50<00:00,  6.05s/it]


Zero shot with gpt-4-turbo


100%|██████████| 48/48 [03:52<00:00,  4.85s/it]


Zero shot with gpt-3.5-turbo


100%|██████████| 48/48 [02:12<00:00,  2.77s/it]


In [4]:
! pip3 install -U tree-of-thoughts

  Using cached httpx-0.27.2-py3-none-any.whl.metadata (7.1 kB)
Using cached httpx-0.27.2-py3-none-any.whl (76 kB)
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.43.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
langchain-anthropic 0.3.10 requires langchain-core<1.0.0,>=0.3.45, but you have langchain-core 1.0.0 which is incompatible.
llama-index-multi-modal-llms-openai 0.4.3 requires llama-index-core<0.13.0,>=0.12.3, but you have llama-index-core 0.14.4 which is incompatible.
llama-index-multi-modal-llms-openai 0.4.3 requires llama-index-llms-openai<0.4.0,>=0.3.0, but you have llama-index-llms-openai 0.6.4 which is incompatible.
llama-index-program-openai 0.3.1 re

In [9]:
from tree_of_thoughts import TotAgent, ToTDFSAgent
from dotenv import load_dotenv

load_dotenv()

# Create an instance of the TotAgent class
tot_agent = TotAgent(use_openai_caller=True)  # Use openai caller

# Create an instance of the ToTDFSAgent class with specified parameters
dfs_agent = ToTDFSAgent(
    agent=tot_agent,  # Use the TotAgent instance as the agent for the DFS algorithm
    threshold=0.8,  # Set the threshold for evaluating the quality of thoughts
    max_loops=1,  # Set the maximum number of loops for the DFS algorithm
    prune_threshold=0.5,  # Branches with evaluation < 0.5 will be pruned
    number_of_agents=4,  # Set the number of agents to be used in the DFS algorithm
)

text="""Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer file. 
The broker is registered in the system so that when a customer calls, based on the contract,
the help desk can immediately trace who is the customer first account manager.
After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, 
so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. 
After the customer profile has been assessed and the customer has been deemed trustworthy,
a preliminary contract/offer on an insurance product is made to the customer either in person or by email. 
(Such offers can also be extended to already existing customers.) If the customer agrees to the offer, 
the contract is signed by both parties. After the signing of the contract,
the client enjoys the coverage and is invoiced (monthly or yearly, depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. 
Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). 
Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise.
According to the reports issued by the estimators, it is decided whether the claim case is approved.
In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  
For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer account. 
The estimators reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 
"""

prompt = f"""
You will be asked by the user to create a plant UML model from specification text. Do so in the most
clear way possible, avoid class properties and assign multiplicity.

Do not include attributes for classes. For example the class Book would be:

class Book{{}}

Use only bi-directional arc for relations and no description. For example a relation between
the class Book and the class Page, if the Book can have from one to many pages and the 
pages could have exactly one book, would be:

Book "1..1" -- "1..*" Page

Adapt the cardinality to each case.

The plantuml has to be the class diagram. In generating the diagram perform this steps in order

1. Extract class from text
2. Extract relations form text
3. Assign the relation to the corresponding class
4. Add cardinality to the relations

Put everything in this order: first all classes and then all relations. In our example would be:

@startuml

class Book{{}}
class Page{{}}

Book "1..1" -- "1..*" Page

@enduml

Output plantuml without futher text or explaination.

##############

The specification text is:

{text}

##############

The uml output is:"""
# Define the initial state for the DFS algorithm
initial_state = prompt

# Run the DFS algorithm to solve the problem and obtain the final thought
final_thought = dfs_agent.run(initial_state)

# Print the final thought in JSON format for easy reading
print(final_thought)



2025-10-25 11:26:43 | WARNING  | swarms.structs.agent:reliability_check:1691 - The model 'None' may not be supported. Please use a supported model, or override the model name with the 'llm' parameter, which should be a class with a 'run(task: str)' method or a '__call__' method.
2025-10-25 11:26:43 | INFO     | tree_of_thoughts.dfs:dfs:57 - Starting DFS for state: 
You will be asked by the user to create a plant UML model from specification text. Do so in the most
clear way possible, avoid class properties and assign multiplicity.

Do not include attributes for classes. For example the class Book would be:

class Book{}

Use only bi-directional arc for relations and no description. For example a relation between
the class Book and the class Page, if the Book can have from one to many pages and the 
pages could have exactly one book, would be:

Book "1..1" -- "1..*" Page

Adapt the cardinality to each case.

The plantuml has to be the class diagram. In generating the diagram perform thi

SyntaxError: invalid syntax (<string>, line 2)

# Zero Shot Open LLM

In [ ]:
import torch, gc

## Deepseek

In [ ]:
from langchain_deepseek import ChatDeepSeek

In [ ]:
MODEL_DEEPSEEK = ["deepseek-chat"]#, "deepseek-reasoner"]

In [ ]:
def deepseek_make_example(model):
    model = llm = ChatDeepSeek(
            model=model,
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            # api_key="...",
            # other params...
        )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [ ]:
def deepseek_zero(model):
    model_ = ChatDeepSeek(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
%%time
display_markdown(deepseek_make_example(MODEL_DEEPSEEK[0]), raw=True)

ValidationError: 1 validation error for ChatDeepSeek
  Value error, If using default api base, DEEPSEEK_API_KEY must be set. [type=value_error, input_value={'model': 'deepseek-chat'...: 2, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error

In [ ]:
for model in MODEL_DEEPSEEK:
    print(f"Zero shot with {model}")
    deepseek_zero(model)

Zero shot with deepseek-chat


ValidationError: 1 validation error for ChatDeepSeek
  Value error, If using default api base, DEEPSEEK_API_KEY must be set. [type=value_error, input_value={'model': 'deepseek-chat', 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error

## Sonnet

In [ ]:
from langchain_anthropic import ChatAnthropic

In [ ]:
MODEL_ANTHROPIC = ["claude-sonnet-4-5-20250929","claude-3-7-sonnet-20250219"]

In [ ]:
def anthropic_make_example(model):
    model = ChatAnthropic(model=model,temperature=0,
    max_tokens=4096,
    timeout=None,
    max_retries=2,)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [ ]:
def anthropic_zero(model):
    model_ = ChatAnthropic(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
display_markdown(anthropic_make_example(MODEL_ANTHROPIC[0]), raw=True)

@startuml

class Customer{}
class Broker{}
class InsurancePolicy{}
class Contract{}
class Offer{}
class Invoice{}
class Claim{}
class ClaimCase{}
class Estimator{}
class Report{}
class CompensationDecision{}
class Payment{}

Customer "1..1" -- "1..1" Broker
Customer "1..1" -- "0..*" InsurancePolicy
Customer "1..1" -- "0..*" Contract
Customer "1..1" -- "0..*" Offer
Customer "1..1" -- "0..*" Invoice
Customer "1..1" -- "0..*" Claim
Customer "1..1" -- "0..*" Payment
Contract "1..1" -- "1..*" InsurancePolicy
Contract "0..1" -- "1..1" Offer
Contract "1..1" -- "0..*" Invoice
Claim "1..1" -- "1..*" ClaimCase
ClaimCase "1..1" -- "1..*" Estimator
Estimator "1..1" -- "1..*" Report
ClaimCase "1..1" -- "0..1" CompensationDecision
CompensationDecision "1..1" -- "1..1" Payment
Report "1..*" -- "1..1" Payment

@enduml

In [ ]:
for model in MODEL_ANTHROPIC:
    print(f"Zero shot with {model}")
    anthropic_zero(model)

Zero shot with claude-sonnet-4-5-20250929


100%|██████████| 48/48 [03:55<00:00,  4.92s/it]


Zero shot with claude-3-7-sonnet-20250219


100%|██████████| 48/48 [02:49<00:00,  3.53s/it]


## Ollama

In [ ]:
from langchain_ollama import ChatOllama

In [ ]:
MODEL_OLLAMA = [] #["llama3.2:3b-text-fp16"]

In [ ]:
def ollama_zero(model):
    model_ = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
def ollama_make_example(model):
    model = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res
    

In [ ]:
#display_markdown(ollama_make_example(MODEL_OLLAMA[0]), raw=True)

In [ ]:
for model in MODEL_OLLAMA:
    print(f"Zero shot with {model}")
    ollama_zero(model)
    gc.collect()
    torch.mps.empty_cache()

## Huggingface

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace, HuggingFaceEndpoint
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/Users/marcocalamo/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/marcocalamo/anaconda3/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <9DBE5D5C-AC87-30CA-96DA-F5BC116EDA2B> /Users/marcocalamo/anaconda3/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <A51C8C05-245A-3989-8D3C-9A6704422CA5> /Users/marcocalamo/anaconda3/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
MODEL_HUGGINGFACE = [] #["Qwen/Qwen2.5-3B-Instruct", "microsoft/Phi-3-mini-4k-instruct", "google/gemma-2-27b-it"]

In [ ]:
def huggingface_zero(model):
    model_id = model
    
    llm = HuggingFaceEndpoint(
        repo_id=model_id,
        task="text-generation",
        max_new_tokens=2048,
        do_sample=False,
        repetition_penalty=1.03,
        temperature=0.01,
    )

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model_id.replace("/","_"))

In [ ]:
def huggingface_make_example(model):

    model_id = model
    
    

    llm = HuggingFacePipeline.from_model_id(
        model_id=model_id,
        task="text-generation",
        pipeline_kwargs={"temperature": 0.1, "max_new_tokens": 1024}
    )

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
        As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.
        
        In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 
        
        """})
    return res


In [ ]:
#display_markdown(huggingface_make_example(MODEL_HUGGINGFACE[0]), raw=True)

In [ ]:
for model in MODEL_HUGGINGFACE:
    print(f"Zero shot with {model}")
    huggingface_zero(model)
    gc.collect()
    torch.mps.empty_cache()

## Mlx-LLM

In [ ]:
from langchain_community.llms import MLXPipeline
from langchain_community.chat_models import ChatMLX

In [ ]:
MODEL_MLX = ["mlx-community/phi-4-8bit",
             "mlx-community/Falcon3-10B-Instruct-8bit", 
             "mlx-community/Qwen2.5-14B-Instruct-4bit",
             "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
             "mlx-community/DeepSeek-R1-Distill-Qwen-7B-8bit",
             "mlx-community/Llama-3.2-3B-Instruct",
             "mlx-community/gemma-2-9b-8bit",
             #"mlx-community/gemma-2-27b-it-4bit",
            # "mlx-community/Mamba-Codestral-7B-v0.1-8bit",
            #"mlx-community/CodeLlama-13b-Instruct-hf-4bit-MLX"
            ]

In [ ]:
def mlx_zero(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 15_000, "temp": 0.1},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model.replace("/","_"))

In [ ]:
def mlx_make_example(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 2000, "temp": 0.7},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer's file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer's profile on the spot or send the customer's file for analysis to the head office. After the customer's profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer's account.  The estimators' reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res


In [ ]:
display_markdown(mlx_make_example(MODEL_MLX[2]), raw=True)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

@startuml

class Customer{}
class Broker{}
class Contract{}
class InsurancePolicy{}
class Claim{}
class Estimator{}
class CompensationDecision{}
class Document{}

Customer -- Broker
Customer -- Contract
Contract -- InsurancePolicy
Customer -- Claim
Claim -- Estimator
Claim -- CompensationDecision
CompensationDecision -- Document

@enduml

In [ ]:
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

In [ ]:
for model in MODEL_MLX[2:3]:
    print(f"Zero shot with {model}")
    mlx_zero(model)
    import gc, torch
    gc.collect()
    torch.mps.empty_cache()

Zero shot with mlx-community/Qwen2.5-14B-Instruct-4bit


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

100%|███████████████████████████████████████████| 48/48 [04:10<00:00,  5.21s/it]


## Clean Cache

In [ ]:
import gc, torch
gc.collect()
torch.mps.empty_cache()